In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_7.py ──
"""
Shared infrastructure for Exercise 7 — Transfer Learning.

Contains: CIFAR-10 data loading, feature visualisation helpers,
ExperimentTracker/ModelRegistry setup, training harness.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex7_transfer_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — CIFAR-10 (full 50K, resized for ResNet-18)
# ════════════════════════════════════════════════════════════════════════
# ResNet-18 was designed for 224x224 ImageNet images. CIFAR-10 is 32x32.
# We resize to 96x96: ResNet's strided convolutions shrink spatial dims
# by 32x, so 32x32 would collapse to 1x1 before final pooling. 96x96
# gives a 3x3 final feature map — enough spatial information to learn.

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SIZE = 96
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Standard transforms: ImageNet normalisation + resize for ResNet
train_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.RandomHorizontalFlip(),
        T.RandomCrop(INPUT_SIZE, padding=8),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
val_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def load_cifar10() -> tuple[
    torchvision.datasets.CIFAR10,
    torchvision.datasets.CIFAR10,
    DataLoader,
    DataLoader,
]:
    """Load CIFAR-10 with ImageNet-style transforms.

    Returns:
        train_set, val_set, train_loader, val_loader
    """
    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=train_transform,
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=val_transform,
    )

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, num_workers=0)

    print(
        f"CIFAR-10 (full): train={len(train_set)}, val={len(val_set)}, "
        f"classes={N_CLASSES}"
    )
    print(f"  Input size: {INPUT_SIZE}x{INPUT_SIZE} (resized for ResNet-18)")
    print(f"  Classes: {CLASS_NAMES}")

    return train_set, val_set, train_loader, val_loader


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_transfer.db"
    registry_db = "sqlite:///mlfp05_transfer_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_transfer_learning", registry, True


def init_engines() -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS — shared by all technique files
# ════════════════════════════════════════════════════════════════════════


async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Train a model and log everything to ExperimentTracker.

    Returns:
        train_losses, val_accs, train_accs (per-epoch)
    """
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"\n-- {name} --  trainable params: {n_trainable:,} / {n_total:,}")

    opt = torch.optim.Adam(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    train_losses: list[float] = []
    val_accs: list[float] = []
    train_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "trainable_params": str(n_trainable),
                "total_params": str(n_total),
                "epochs": str(epochs),
                "lr": str(lr),
                "batch_size": str(tr_loader.batch_size),
                "dataset_size": str(len(tr_loader.dataset)),
            }
        )

        for epoch in range(epochs):
            # -- Training --
            model.train()
            batch_losses = []
            correct = total = 0
            for xb, yb in tr_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
                correct += int((logits.argmax(dim=-1) == yb).sum().item())
                total += int(yb.size(0))
            train_losses.append(float(np.mean(batch_losses)))
            train_accs.append(correct / total)
            scheduler.step()

            # -- Validation --
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for xb, yb in vl_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    preds = model(xb).argmax(dim=-1)
                    correct += int((preds == yb).sum().item())
                    total += int(yb.size(0))
            val_accs.append(correct / total)

            await run.log_metrics(
                {
                    "train_loss": train_losses[-1],
                    "train_acc": train_accs[-1],
                    "val_acc": val_accs[-1],
                },
                step=epoch + 1,
            )

            print(
                f"  epoch {epoch + 1}/{epochs}  "
                f"loss={train_losses[-1]:.4f}  "
                f"train_acc={train_accs[-1]:.3f}  "
                f"val_acc={val_accs[-1]:.3f}"
            )

        await run.log_metrics(
            {
                "final_val_acc": val_accs[-1],
                "best_val_acc": max(val_accs),
                "final_train_loss": train_losses[-1],
            }
        )

    return train_losses, val_accs, train_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            tr_loader,
            vl_loader,
            epochs,
            lr,
        )
    )


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Register a trained model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="val_acc", value=val_acc),
            MetricSpec(name="final_loss", value=final_loss),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={val_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, val_acc, final_loss))


# ════════════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION & VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════


def extract_features(
    model: nn.Module,
    loader: DataLoader,
    max_samples: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    """Extract features from the penultimate layer (before fc head).

    Works for both ResNet (avgpool hook) and Sequential models.
    """
    model.eval()
    hook_features: list[torch.Tensor] = []
    labels: list[np.ndarray] = []

    def hook_fn(module, inp, out):
        del module, inp  # PyTorch hook protocol args; not used here
        hook_features.append(out.flatten(1).detach().cpu())

    # ResNet: hook into avgpool; Sequential: second-to-last layer
    if hasattr(model, "avgpool"):
        avgpool = model.avgpool
        assert isinstance(
            avgpool, nn.Module
        ), f"expected nn.Module for avgpool, got {type(avgpool).__name__}"
        handle = avgpool.register_forward_hook(hook_fn)
    else:
        assert isinstance(
            model, nn.Sequential
        ), f"hook fallback requires nn.Sequential, got {type(model).__name__}"
        handle = model[-3].register_forward_hook(hook_fn)

    with torch.no_grad():
        collected = 0
        for xb, yb in loader:
            if collected >= max_samples:
                break
            xb = xb.to(device)
            model(xb)
            labels.append(yb.numpy())
            collected += len(yb)

    handle.remove()
    features_np = torch.cat(hook_features, dim=0).numpy()[:max_samples]
    labels_np = np.concatenate(labels)[:max_samples]
    return features_np, labels_np


def compute_tsne(features: np.ndarray, perplexity: int = 30) -> np.ndarray:
    """Run t-SNE dimensionality reduction to 2D."""
    # sklearn 1.5+ renamed n_iter → max_iter in TSNE.__init__
    tsne = TSNE(n_components=2, perplexity=perplexity, max_iter=500, random_state=42)
    return tsne.fit_transform(features)


def cluster_quality(coords: np.ndarray, labels: np.ndarray) -> float:
    """Cluster quality: ratio of intra-class to inter-class distance (lower = better)."""
    intra = []
    centroids = []
    for c in range(N_CLASSES):
        mask = labels == c
        if mask.sum() < 2:
            continue
        pts = coords[mask]
        centroid = pts.mean(axis=0)
        centroids.append(centroid)
        intra.append(np.mean(np.linalg.norm(pts - centroid, axis=1)))
    centroids_arr = np.array(centroids)
    inter = np.mean(
        [
            np.linalg.norm(centroids_arr[i] - centroids_arr[j])
            for i in range(len(centroids_arr))
            for j in range(i + 1, len(centroids_arr))
        ]
    )
    avg_intra = np.mean(intra)
    return float(avg_intra / inter) if inter > 0 else float("inf")


def plot_tsne(
    coords: np.ndarray,
    labels: np.ndarray,
    title: str,
    output_path: str | Path,
) -> None:
    """Create and save a t-SNE scatter plot coloured by class."""
    fig = go.Figure()
    for c in range(N_CLASSES):
        mask = labels == c
        fig.add_trace(
            go.Scatter(
                x=coords[mask, 0],
                y=coords[mask, 1],
                mode="markers",
                name=CLASS_NAMES[c],
                marker=dict(size=4, opacity=0.6),
            )
        )
    fig.update_layout(
        title=title,
        xaxis_title="t-SNE 1",
        yaxis_title="t-SNE 2",
        template="plotly_white",
    )
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance."""
    return ModelVisualizer()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(metrics=metrics, x_label="Epoch", y_label="Value")
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def count_params(model: nn.Module, trainable_only: bool = False) -> int:
    """Count parameters in a model."""
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 7, Part 1: Training a CNN from Scratch (Baseline)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this section, you will be able to:
#   - Build and train a CNN from random initialisation on CIFAR-10
#   - Understand why training from scratch requires massive labelled data
#   - Visualise learned filters at different layers (random early vs
#     structured later) and feature representations with t-SNE
#   - Quantify the data bottleneck that motivates transfer learning
#
# PREREQUISITES: M5/ex_2 (CNNs and PyTorch), M5/ex_1 (ExperimentTracker).
# ESTIMATED TIME: ~25 min
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn



## THEORY — Why Training from Scratch Is Hard


In [ ]:
# A CNN initialised with random weights knows nothing about the visual
# world. Every filter — edge detectors, texture recognisers, shape
# combinations — must be learned entirely from your training data.
#
# The ImageNet revolution (2012) showed that deep CNNs can learn these
# filters, but they needed 1.28 MILLION labelled images to do it.
# Most real-world projects have 1,000 - 50,000 labelled images, not
# millions. This creates the "data bottleneck": the model has enough
# capacity to learn, but not enough examples to learn FROM.
#
# This baseline quantifies the problem. We train a small CNN from
# scratch on CIFAR-10 (50K images) and measure its accuracy. In the
# next section, we'll compare this against a ResNet-18 that starts
# with ImageNet knowledge — and the gap will motivate everything that
# follows.



In [ ]:
print("\n" + "=" * 70)
print("  PART 1: Training a CNN from Scratch (Baseline)")
print("=" * 70)



## TASK 1 — Load CIFAR-10 and initialise kailash-ml engines


In [ ]:
train_set, val_set, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()



### Checkpoint 1


In [ ]:
assert len(train_set) == 50000, (
    f"Expected full 50K CIFAR-10, got {len(train_set)}. "
    "Transfer learning exercises need the full dataset."
)
assert len(val_set) == 10000, "CIFAR-10 test set should be 10K"
assert tracker is not None, "ExperimentTracker should be initialised"
# INTERPRETATION: We use the full 50K training set so that the
# from-scratch baseline has every possible advantage. Even with all
# 50K images, the scratch CNN will be limited because it must learn
# every visual feature from these images alone.
print("\n--- Checkpoint 1 passed --- CIFAR-10 loaded, engines ready\n")



## TASK 2 — Build a from-scratch CNN


Baseline: a small CNN trained from random init.


In [ ]:
# A small CNN trained from random initialisation. We keep the
# architecture simple (3 conv layers + classifier) to isolate the
# effect of pre-training vs architecture advantage.


def build_scratch_cnn(n_classes: int = N_CLASSES) -> nn.Module:
    # TODO: Build a nn.Sequential CNN with:
    #   - Conv2d(3, 32, 3, padding=1) + BatchNorm2d(32) + ReLU + MaxPool2d(2)
    #   - Conv2d(32, 64, 3, padding=1) + BatchNorm2d(64) + ReLU + MaxPool2d(2)
    #   - Conv2d(64, 128, 3, padding=1) + BatchNorm2d(128) + ReLU + AdaptiveAvgPool2d(1)
    #   - Flatten + Dropout(0.3) + Linear(128, n_classes)
    # Hint: nn.Sequential takes all layers as positional arguments
    pass  # Replace with your implementation


scratch_model = build_scratch_cnn()
n_total = count_params(scratch_model)
print(f"  From-scratch CNN: {n_total:,} total parameters (all trainable)")



### Checkpoint 2


In [ ]:
assert n_total > 50000, f"CNN should have >50K params, got {n_total:,}"
assert n_total < 500000, f"CNN should be small (<500K params), got {n_total:,}"
# INTERPRETATION: This is a deliberately small CNN. We keep it small
# so the comparison with transfer learning is about knowledge, not
# architecture size. The scratch CNN has to learn edge detectors,
# texture recognisers, and shape combinations all from 50K images.
print("--- Checkpoint 2 passed --- CNN architecture built\n")



## TASK 3 — Train the from-scratch CNN with ExperimentTracker


In [ ]:
print("\n" + "=" * 70)
print("  TRAINING: From-Scratch CNN on full CIFAR-10 (50K)")
print("=" * 70)

scratch_losses, scratch_accs, scratch_train_accs = train_model(
    scratch_model,
    "cnn_from_scratch",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)

best_scratch = max(scratch_accs)



### Checkpoint 3


In [ ]:
assert len(scratch_losses) == EPOCHS, "Should have per-epoch losses"
assert (
    best_scratch > 0.30
), f"Scratch val accuracy {best_scratch:.3f} below 0.30 -- check training"
# INTERPRETATION: With 50K images and 8 epochs, the scratch CNN
# reaches a reasonable but not stellar accuracy. Every percentage point
# required the model to discover visual features from raw pixels.
# In part 2, we'll see how much ImageNet pre-training changes this.
print(f"\n  From-scratch best val_acc: {best_scratch:.3f}")
print("--- Checkpoint 3 passed --- baseline training complete\n")



## TASK 4 — Visualise: Learned filters and t-SNE feature space


In [ ]:
# Two visualisations reveal what the scratch CNN actually learned:
#
# 1. LEARNED FILTERS — The first conv layer's 32 filters. In a
#    randomly-initialised network, these look like noise. After
#    training, some develop into crude edge/colour detectors, but
#    they're much noisier than pre-trained ResNet filters.
#
# 2. t-SNE FEATURE SPACE — We extract the 128-dimensional features
#    from the penultimate layer and project to 2D. If the CNN learned
#    good representations, classes should cluster. Scratch-trained
#    features typically show more overlap between classes.

# -- Learned filters visualisation --
print("-- Visualising learned convolutional filters --")

# TODO: Extract first conv layer weights and visualise them
# Steps:
#   1. Get weights: scratch_model[0].weight.data.cpu()  -> shape (32, 3, 3, 3)
#   2. Pick first 16 filters (n_filters = min(16, ...))
#   3. For each filter: normalise to [0,1], average across RGB channels
#   4. Create a go.Figure() with go.Heatmap traces for each filter
#   5. Save to OUTPUT_DIR / "01_scratch_filters.html"
# Hint: filt_norm = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
# Hint: filt_gray = filt_norm.mean(dim=0).numpy()

conv1_weights = scratch_model[0].weight.data.cpu()  # shape: (32, 3, 3, 3)
n_filters = min(16, conv1_weights.shape[0])

fig_filters = go.Figure()
# TODO: Loop through n_filters, normalise each filter, add Heatmap traces
# Hint: Use np.flipud() for display, colorscale="Greys"
pass  # Replace with your filter visualisation loop

fig_filters.update_layout(
    title="From-Scratch CNN: Learned First-Layer Filters (noisy, unstructured)",
    template="plotly_white",
    width=600,
    height=600,
)
filters_path = OUTPUT_DIR / "01_scratch_filters.html"
fig_filters.write_html(str(filters_path))
print(f"  Saved: {filters_path}")

# -- t-SNE feature space --
print("-- Extracting features for t-SNE visualisation --")
scratch_feats, scratch_labels = extract_features(scratch_model, val_loader)
print(f"  Scratch features shape: {scratch_feats.shape}")

coords_scratch = compute_tsne(scratch_feats)
cq_scratch = cluster_quality(coords_scratch, scratch_labels)

print(f"  t-SNE cluster quality (lower = better): {cq_scratch:.4f}")
print(f"\n  Class centroids (first 5 classes):")
for c in range(min(5, N_CLASSES)):
    mask = scratch_labels == c
    centroid = coords_scratch[mask].mean(axis=0)
    print(f"    {CLASS_NAMES[c]:>12}: ({centroid[0]:+.1f}, {centroid[1]:+.1f})")

tsne_path = OUTPUT_DIR / "01_scratch_tsne.html"
plot_tsne(coords_scratch, scratch_labels, "t-SNE: From-Scratch CNN Features", tsne_path)

# Save training curves
viz = create_visualizer()
save_training_plots(
    viz,
    {"scratch loss": scratch_losses, "scratch val_acc": scratch_accs},
    OUTPUT_DIR / "01_scratch_training.html",
)



### Checkpoint 4


In [ ]:
assert scratch_feats.shape[0] > 0, "Should have extracted features"
assert coords_scratch.shape == (scratch_feats.shape[0], 2), "t-SNE should produce 2D"
# INTERPRETATION: The t-SNE plot likely shows overlapping clusters —
# classes that the scratch CNN can't fully separate. This is what
# "learning from scratch" looks like: with only 50K images and 8
# epochs of training, the feature representations are noisy and
# incomplete. Compare this to the transfer learning t-SNE in part 2.
print("\n--- Checkpoint 4 passed --- visualisations complete\n")



## TASK 5 — Apply: Small Singapore Startup with Limited Data


In [ ]:
# SCENARIO: You're the ML engineer at a Singapore startup (like Carro
# or ShopBack) trying to build an image classifier for product photos.
# You only have 5,000 labelled images — 10% of CIFAR-10.
#
# Question: What happens to from-scratch accuracy with limited data?

print("\n" + "=" * 70)
print("  APPLY: Singapore Startup with 5,000 Labelled Images")
print("=" * 70)

# TODO: Simulate the startup scenario by training on only 10% of CIFAR-10
# Steps:
#   1. Create a random subset of 5,000 indices from train_set
#   2. Build a Subset and DataLoader from those indices
#   3. Train a fresh build_scratch_cnn() on the limited data
#   4. Compare best accuracy with the full-data result
# Hint: rng = np.random.default_rng(42)
# Hint: indices = rng.choice(len(train_set), size=5000, replace=False).tolist()
# Hint: Use Subset(train_set, indices) and DataLoader

rng = np.random.default_rng(42)
n_startup = 5000
# TODO: Create indices, subset, and loader
# TODO: Build and train a fresh scratch CNN on the limited data
# TODO: Store the best accuracy in best_startup
from torch.utils.data import Subset, DataLoader as DL

indices = (
    None  # TODO: rng.choice(len(train_set), size=n_startup, replace=False).tolist()
)
startup_subset = None  # TODO: Subset(train_set, indices)
startup_loader = (
    None  # TODO: DL(startup_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
)

startup_model = build_scratch_cnn()
# TODO: Train startup_model using train_model() with startup_loader
# startup_losses, startup_accs, _ = train_model(...)
startup_losses, startup_accs = [], []  # Replace with actual training
best_startup = 0.0  # TODO: max(startup_accs) after training

print(f"\n  === Startup Scenario Results ===")
print(f"  Full data (50K):    {best_scratch:.1%} accuracy")
print(f"  Startup (5K):       {best_startup:.1%} accuracy")
print(f"  Accuracy drop:      {best_scratch - best_startup:+.1%}")
print()
print(f"  BUSINESS IMPACT:")
print(f"  With only 5,000 labelled images, from-scratch training loses")
print(f"  significant accuracy. For a product classifier at a Singapore")
print(f"  startup, this means:")
print(f"    - More misclassified products shown to customers")
print(f"    - Higher rate of manual review needed")
print(f"    - Labelling 50K images costs ~S$25,000-50,000 (S$0.50-1.00/label)")
print(f"  This is exactly the problem transfer learning solves (Part 2).")



### Checkpoint 5


In [ ]:
assert (
    best_startup > 0.15
), f"Even with 5K data, should beat random chance (acc={best_startup:.3f})"
assert (
    best_startup < best_scratch
), "With less data, accuracy should be lower than full-data training"
# INTERPRETATION: The accuracy drop from 50K to 5K images quantifies
# the data bottleneck. This is the core motivation for transfer
# learning: instead of collecting 10x more labelled data (expensive),
# we can reuse features already learned from ImageNet (free).
print("\n--- Checkpoint 5 passed --- startup scenario complete\n")



## CLEANUP


In [ ]:

await conn.close()



## REFLECTION


[x] Built and trained a from-scratch CNN on CIFAR-10 ({n_total:,} params)
  [x] Achieved {best_scratch:.1%} val accuracy with full 50K training data
  [x] Visualised learned filters (noisy, unstructured after 8 epochs)
  [x] Mapped feature space with t-SNE (cluster quality: {cq_scratch:.3f})
  [x] Quantified the startup data bottleneck: {best_startup:.1%} with only 5K images

  KEY INSIGHT: Training from scratch requires massive labelled datasets.
  A Singapore startup with 5,000 images loses significant accuracy because
  the CNN must discover every visual feature from limited examples.

  NEXT: Part 2 uses a pre-trained ResNet-18 backbone to solve this exact
  problem. The pre-trained model brings ImageNet knowledge (edges, textures,
  shapes) and only needs to learn the final classification layer.


In [ ]:
print("\n" + "=" * 70)
print("  PART 1 COMPLETE — What You've Learned")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `kailash_ml.diagnostics` (via `kailash-ml`) — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    # Classification CE on Fashion-MNIST from scratch
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (From-scratch CNN baseline (no pretrain)) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        model,
        train_loader,
        _diag_loss,
        title="From-scratch CNN baseline (no pretrain)",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [!] Gradient flow (WARNING): RMS 2.1e-05 in early conv layers (near vanishing).
# [!] Dead neurons  (WARNING): 43% inactive in conv1 — classic ReLU-dead-from-scratch.
# [✓] Loss trend    (HEALTHY): slowly converging, plateau risk.



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [BLOOD TEST] Training from scratch hits the classic early-layer
#     vanishing gradient. Without pretrained weights, early conv
#     filters start random and receive weak gradient signal.
#     >> This is THE reason transfer learning (ex_7/02) wins —
#        starting with ImageNet features means first-layer gradients
#        are already ~10x larger than this.
#
#  [X-RAY] 43% dead ReLU on conv1 is the dying-ReLU failure mode
#     slide 5.7 warns about. Fix: GELU or Kaiming init.
#     >> Prescription: replace ReLU with GELU, use Kaiming init,
#        OR apply transfer learning (skip this problem entirely).
#
#  [STETHOSCOPE] Slow convergence (< 60% accuracy in 10 epochs)
#     — expect transfer learning to reach 85%+ in the same time.

